In [1]:
import numpy as np
import pandas as pd

In [2]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder

In [3]:
df=pd.read_csv("../data/covid_toy.csv")

In [4]:
df.head()

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


In [5]:
df["cough"].value_counts()

cough
Mild      62
Strong    38
Name: count, dtype: int64

In [6]:
df["city"].value_counts()

city
Kolkata      32
Bangalore    30
Delhi        22
Mumbai       16
Name: count, dtype: int64

In [7]:
df.isnull().sum()

age           0
gender        0
fever        10
cough         0
city          0
has_covid     0
dtype: int64

In [9]:
X=df.drop("has_covid",axis=1)
y=df["has_covid"]

In [13]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [17]:
#simple imputer for fever cols
si=SimpleImputer()
X_train_fever=si.fit_transform(X_train[['fever']])
X_test_fever=si.fit_transform(X_test[['fever']])

In [16]:
X_train_fever.shape

(80, 1)

In [19]:
#ordinal encoding ->cough
oe = OrdinalEncoder(categories=[['Mild', 'Strong']])

X_train_cough = oe.fit_transform(X_train[['cough']])
X_test_cough = oe.transform(X_test[['cough']])

In [20]:
X_train_cough.shape

(80, 1)

In [31]:
#one hot encoding->gender,city
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore',drop='first')

X_train_ohe = ohe.fit_transform(X_train[['gender', 'city']])
X_test_ohe = ohe.transform(X_test[['gender', 'city']])

In [32]:
X_train_ohe.shape

(80, 4)

In [33]:
#extracting age
X_train_age=X_train.drop(columns=['gender','fever','cough','city']).values
X_test_age=X_test.drop(columns=['gender','fever','cough','city']).values

In [34]:
X_train_age.shape

(80, 1)

In [35]:
X_train_transformed=np.concatenate((X_train_age,X_train_ohe,X_train_cough,X_train_fever),axis=1)
X_test_transformed=np.concatenate((X_test_age,X_test_ohe,X_test_cough,X_test_fever),axis=1)

In [37]:
X_train_transformed.shape

(80, 7)

# Using column transformer

In [38]:
from sklearn.compose import ColumnTransformer

In [42]:


transformer = ColumnTransformer(transformers=[
    ('tnf1', SimpleImputer(), ['fever']),
    ('tnf2', OrdinalEncoder(categories=[['Mild', 'Strong']]), ['cough']),
    ('tnf3', OneHotEncoder(sparse_output=False, drop='first'), ['gender', 'city'])
], remainder='passthrough')

In [45]:
transformer.fit_transform(X_train).shape

(80, 7)

In [47]:
transformer.transform(X_test)

array([[104.,   0.,   0.,   0.,   1.,   0.,  17.],
       [ 98.,   0.,   1.,   1.,   0.,   0.,  83.],
       [101.,   1.,   0.,   1.,   0.,   0.,  68.],
       [ 99.,   0.,   1.,   0.,   0.,   0.,  72.],
       [102.,   1.,   1.,   1.,   0.,   0.,  20.],
       [103.,   0.,   0.,   0.,   1.,   0.,  50.],
       [ 98.,   1.,   0.,   0.,   1.,   0.,  71.],
       [ 99.,   0.,   0.,   0.,   0.,   1.,  14.],
       [101.,   0.,   0.,   1.,   0.,   0.,  75.],
       [103.,   0.,   1.,   0.,   1.,   0.,  60.],
       [ 98.,   0.,   0.,   0.,   0.,   0.,  64.],
       [101.,   0.,   1.,   1.,   0.,   0.,  15.],
       [ 98.,   1.,   1.,   0.,   1.,   0.,  34.],
       [ 98.,   0.,   0.,   0.,   1.,   0.,  26.],
       [ 99.,   1.,   0.,   1.,   0.,   0.,  59.],
       [101.,   0.,   0.,   0.,   0.,   1.,  65.],
       [100.,   0.,   1.,   0.,   0.,   0.,  80.],
       [101.,   0.,   0.,   0.,   1.,   0.,   8.],
       [ 99.,   1.,   0.,   0.,   1.,   0.,  25.],
       [103.,   0.,   1.,   0.,

In [48]:
transformer.transform(X_test).shape

(20, 7)